# OMI Residential Real Estate Quotations — Dataset Creation & Data Quality

This notebook builds a reproducible, validated analytical dataset from the semiannual
**OMI (Osservatorio del Mercato Immobiliare)** quotations published by the Italian
*Agenzia delle Entrate*. It is the ingestion pipeline that feeds the companion
exploratory-analysis notebook (`01_02_omi_quotations_exploration.ipynb`).

**Source:** Agenzia delle Entrate — Osservatorio del Mercato Immobiliare (OMI), semiannual
quotation releases, distributed as one semicolon-separated CSV per semester.

## Analytical objective

Move from raw OMI releases to a validated, typed, analysis-ready dataset that can
support: temporal market analysis, geographic comparisons, quotation-range analysis,
property-category analysis, and future joins with transaction-volume data.

## Design principles

- **Conservative cleaning.** No quotation values are imputed. Genuine zero values are
  distinguished from placeholder zeros; ambiguous or invalid records are *flagged*,
  not silently dropped, so downstream notebooks can decide how to treat them.
- **Every transformation is measured.** Whenever a value is coerced, replaced or
  dropped, the number of affected rows is reported, so silent data loss is never
  invisible.
- **Symmetric treatment of purchase and rental fields.** Any rule applied to the
  purchase quotation (`Compr_*`) is applied identically to the rental quotation
  (`Loc_*`), unless there is an explicit, documented reason not to.

## Contents

1. Setup
2. Data Loading
3. Initial Dataset Inspection
4. Data Quality Assessment
5. Structural Validation
6. Cleaning & Standardisation
7. Feature Engineering
8. Temporal Coverage
9. Geographic Coverage
10. Property Categories & Conditions
11. Final Data-Quality Summary
12. Export to Parquet


## 1. Setup

The analytical stack is kept lightweight on purpose: `pandas` for data preparation,
`numpy` for numerical transformations and `matplotlib` for visualization. The project
root is resolved relative to the notebook's working directory so the code does not
depend on where the notebook happens to be launched from.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'quotations'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

assert RAW_DIR.exists(), f'Raw quotations directory not found: {RAW_DIR}'

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 2. Data Loading

OMI quotations are supplied as one semicolon-separated CSV per semester, named
`omi_quotations_YYYY_S1.csv` / `omi_quotations_YYYY_S2.csv`. Each file is loaded
independently — rather than with a single glob-and-concat call — so that a file whose
name does not match the expected convention is surfaced explicitly instead of silently
breaking the year/semester tagging that every later section depends on.


In [3]:
files = sorted(RAW_DIR.glob('omi_quotations_*.csv'))

if not files:
    raise FileNotFoundError(f'No OMI quotation files found in {RAW_DIR}')

records = []
invalid_files = []

for path in files:
    parts = path.stem.rsplit('_', 2)

    # Expected format: omi_quotations_YYYY_S1 / omi_quotations_YYYY_S2
    if len(parts) != 3 or not parts[1].isdigit() or parts[2] not in {'S1', 'S2'}:
        invalid_files.append(path.name)
        continue

    year = int(parts[1])
    semester = parts[2]

    df_part = pd.read_csv(path, sep=';', low_memory=False)

    df_part['reference_year'] = year
    df_part['reference_semester'] = semester
    df_part['reference_period'] = f'{year}-{semester}'

    records.append(df_part)

if invalid_files:
    print('Files skipped because their name does not match the expected convention:')
    print(invalid_files)

if not records:
    raise ValueError('No valid OMI quotation files were found.')

omi = pd.concat(records, ignore_index=True)

print(f'Files loaded:      {len(records):,}')
print(f'Rows consolidated: {len(omi):,}')
print(f'Columns:           {omi.shape[1]:,}')


Files loaded:      44
Rows consolidated: 7,516,495
Columns:           25


## 3. Initial Dataset Inspection

Inspect the schema, temporal coverage and raw structure before changing any values.


### 3.1 Raw Preview and Schema

In [4]:
display(omi.head())
display(omi.dtypes.to_frame('dtype'))
print('Raw shape:', omi.shape)


,Area_territoriale,Regione,Prov,Comune_ISTAT,Comune_cat,Sez,Comune_amm,Comune_descrizione,Fascia,Zona,LinkZona,Cod_Tip,Descr_Tipologia,Stato,Stato_prev,Compr_min,Compr_max,Sup_NL_compr,Loc_min,Loc_max,Sup_NL_loc,Unnamed: 21,reference_year,reference_semester,reference_period
0,NORD-OVEST,PIEMONTE,AL,"1,006,003.00",A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,20,Abitazioni civili,NORMALE,,"1,110.00","1,670.00",L,"3,7","5,6",L,NaN,2004,S1,2004-S1
1,NORD-OVEST,PIEMONTE,AL,"1,006,003.00",A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,13,Box,NaN,,"1,010.00","1,520.00",L,"4,3","6,4",L,NaN,2004,S1,2004-S1
2,NORD-OVEST,PIEMONTE,AL,"1,006,003.00",A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,14,Posti auto coperti,NaN,,610.00,840.00,L,"2,5","3,5",L,NaN,2004,S1,2004-S1
3,NORD-OVEST,PIEMONTE,AL,"1,006,003.00",A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,15,Posti auto scoperti,NaN,,490.00,660.00,L,2,"2,7",L,NaN,2004,S1,2004-S1
4,NORD-OVEST,PIEMONTE,AL,"1,006,003.00",A2AA,,A182,ALESSANDRIA,B,B1,AL00000001,9,Magazzini,NORMALE,,"1,180.00","1,410.00",L,"5,4","6,5",L,NaN,2004,S1,2004-S1


,dtype
Area_territoriale,str
Regione,str
Prov,str
Comune_ISTAT,float64
Comune_cat,str
Sez,str
Comune_amm,str
Comune_descrizione,str
Fascia,str
Zona,str


Raw shape: (7516495, 25)


### 3.2 Duplicate Rows

In [5]:
print('Fully duplicated rows:', omi.duplicated().sum())


Fully duplicated rows: 0


### 3.3 Coverage by Reference Period

A quick row-count-per-semester snapshot. This is a lightweight sanity check; the full
continuity check against the expected semester sequence is done in Section 8.


In [6]:
display(
    omi['reference_period']
    .value_counts()
    .sort_index()
    .rename_axis('reference_period')
    .reset_index(name='rows')
)


,reference_period,rows
0,2004-S1,172723
1,2004-S2,174690
2,2005-S1,176215
3,2005-S2,177952
4,2006-S1,179700
5,2006-S2,181054
6,2007-S1,181111
7,2007-S2,181904
8,2008-S1,181768
9,2008-S2,183380


## 4. Data Quality Assessment

Missing values, parsing artefacts and coercion failures are measured before deciding how
to handle them. A missing quotation is not automatically equivalent to a zero
quotation, and a value that fails numeric parsing is not automatically equivalent to a
value that was originally missing — the two are tracked separately below.


### 4.1 Missing Values Overview

In [7]:
missing = (
    omi.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('missing_pct')
    .to_frame()
)

display(missing.head(25))


,missing_pct
Unnamed: 21,100.00
Loc_min,7.28
Loc_max,7.28
Sup_NL_loc,6.95
Prov,2.05
Stato,1.28
Sup_NL_compr,0.02
Compr_min,0.02
Compr_max,0.02
Comune_ISTAT,0.00


### 4.2 Technical / Artefact Columns

CSV exports sometimes carry stray `Unnamed: N` columns produced by trailing delimiters.
Only columns that are **entirely** empty are dropped automatically; any `Unnamed:`
column that does carry data is left in place and flagged for manual inspection, since
silently dropping it could discard real information.


In [8]:
technical_columns = [column for column in omi.columns if column.startswith('Unnamed:')]

technical_check = {
    column: {'all_missing': omi[column].isna().all(), 'non_null': int(omi[column].notna().sum())}
    for column in technical_columns
}

display(pd.DataFrame(technical_check).T)

columns_to_drop = [column for column in technical_columns if omi[column].isna().all()]
columns_kept_for_review = [column for column in technical_columns if column not in columns_to_drop]

omi = omi.drop(columns=columns_to_drop)

print(f'Dropped {len(columns_to_drop)} fully-empty technical column(s): {columns_to_drop}')
if columns_kept_for_review:
    print(f'Kept for manual review (not fully empty): {columns_kept_for_review}')


,all_missing,non_null
Unnamed: 21,True,0


Dropped 1 fully-empty technical column(s): ['Unnamed: 21']


### 4.3 Numeric Quotation Fields

Quotation columns are converted to numeric types explicitly, with `errors='coerce'` so
that malformed values become visible as missing values instead of causing silent
downstream errors. Coercion can itself destroy information silently if a value that was
originally present fails to parse — so we measure exactly how many non-null values were
lost to coercion, per column, rather than assuming the conversion was lossless.


In [9]:
numeric_columns = ['Compr_min', 'Compr_max', 'Loc_min', 'Loc_max']
numeric_columns = [column for column in numeric_columns if column in omi.columns]

coercion_report = {}
for column in numeric_columns:
    non_null_before = omi[column].notna().sum()
    converted = pd.to_numeric(omi[column], errors='coerce')
    non_null_after = converted.notna().sum()
    coercion_report[column] = {
        'non_null_before': int(non_null_before),
        'non_null_after': int(non_null_after),
        'values_lost_to_coercion': int(non_null_before - non_null_after)
    }
    omi[column] = converted

display(pd.DataFrame(coercion_report).T)
display(omi[numeric_columns].describe().T)


,non_null_before,non_null_after,values_lost_to_coercion
Compr_min,7514937,7514937,0
Compr_max,7514937,7514937,0
Loc_min,6969245,1101947,5867298
Loc_max,6969245,1092249,5876996


,count,mean,std,min,25%,50%,75%,max
Compr_min,"7,514,937.00",792.99,618.41,0.00,400.00,630.00,"1,000.00","20,000.00"
Compr_max,"7,514,937.00","1,070.95",848.85,0.00,550.00,850.00,"1,330.00","33,360.00"
Loc_min,"1,101,947.00",3.53,3.02,0.00,2.00,3.00,4.00,105.00
Loc_max,"1,092,249.00",5.02,4.48,0.00,3.00,4.00,6.00,509.00


### 4.4 Zero-Value Handling

A quotation of exactly **€0/m²** is not an economically meaningful purchase or rental
price — in OMI extracts it typically signals an unsurveyed range rather than a genuine
market value of zero. It is therefore treated as missing. This rule is applied
**symmetrically** to both the purchase (`Compr_*`) and rental (`Loc_*`) fields; applying
it to one but not the other would make later purchase/rental comparisons rest on
different missingness conventions.


In [10]:
zero_value_report = {}
for column in numeric_columns:
    zero_count = int(omi[column].eq(0).sum())
    zero_value_report[column] = {'zero_values_nulled': zero_count}
    omi[column] = omi[column].replace(0, np.nan)

display(pd.DataFrame(zero_value_report).T)


,zero_values_nulled
Compr_min,24819
Compr_max,24819
Loc_min,39501
Loc_max,39501


## 5. Structural Validation

The following checks test assumptions that should hold for an OMI quotation record:
minimum values must not exceed maximum values, quotations must not be negative, and the
semester must be valid. Consistent with the conservative-cleaning principle stated in
the introduction, violating rows are **flagged** with boolean indicator columns rather
than dropped or altered — the raw values are preserved, and any downstream analysis can
choose to filter on the flags.


### 5.1 Range Consistency and Sign Checks

In [11]:
omi['flag_compr_min_gt_max'] = omi['Compr_min'] > omi['Compr_max']
omi['flag_loc_min_gt_max'] = omi['Loc_min'] > omi['Loc_max']
omi['flag_negative_compr'] = (omi['Compr_min'] < 0) | (omi['Compr_max'] < 0)
omi['flag_negative_loc'] = (omi['Loc_min'] < 0) | (omi['Loc_max'] < 0)
omi['flag_invalid_semester'] = ~omi['reference_semester'].isin(['S1', 'S2'])

validation = pd.DataFrame({
    'check': [
        'Compr_min > Compr_max',
        'Loc_min > Loc_max',
        'Negative purchase quotation',
        'Negative rental quotation',
        'Invalid semester'
    ],
    'violations': [
        int(omi['flag_compr_min_gt_max'].sum()),
        int(omi['flag_loc_min_gt_max'].sum()),
        int(omi['flag_negative_compr'].sum()),
        int(omi['flag_negative_loc'].sum()),
        int(omi['flag_invalid_semester'].sum())
    ]
})

display(validation)


,check,violations
0,Compr_min > Compr_max,0
1,Loc_min > Loc_max,0
2,Negative purchase quotation,0
3,Negative rental quotation,0
4,Invalid semester,0


### 5.2 Candidate Analytical Key and Duplicates

`(reference_period, Comune_ISTAT, Fascia, Zona, Descr_Tipologia, Stato)` is the natural
analytical key for an OMI record — one quotation per municipality, zone, band, typology,
condition and semester. Rows sharing this key are not removed here (a legitimate OMI
extract can occasionally carry more than one record per key, e.g. across floor-level
sub-bands not captured by this key); the count is reported so that any downstream
aggregation that assumes key-uniqueness is done with that caveat in mind.


In [12]:
candidate_key = ['reference_period', 'Comune_ISTAT', 'Fascia', 'Zona', 'Descr_Tipologia', 'Stato']
available_key = [column for column in candidate_key if column in omi.columns]

duplicate_key_rows = omi.duplicated(subset=available_key, keep=False).sum() if available_key else 0

print('Candidate analytical key:', available_key)
print(f'Rows participating in duplicate candidate keys: {duplicate_key_rows:,} ({duplicate_key_rows / len(omi):.2%} of rows)')


Candidate analytical key: ['reference_period', 'Comune_ISTAT', 'Fascia', 'Zona', 'Descr_Tipologia', 'Stato']
Rows participating in duplicate candidate keys: 1,919 (0.03% of rows)


## 6. Cleaning and Standardisation

Cleaning is conservative: text fields are trimmed, empty strings become missing values,
and only fully-empty technical columns were removed in Section 4.2. Valid zero values
were already handled in Section 4.4. No quotation values are imputed.

**Robustness note.** Text columns are selected with
`select_dtypes(include=['object', 'string'])` rather than `include='str'`: which dtype
name pandas assigns to CSV-derived text columns (`object` vs. the newer `string`/`str`
backend) depends on the pandas version and configuration in use, and selecting only
`'str'` would silently select zero columns — and therefore silently skip all text
cleaning — on any environment where text columns are still `object` dtype.


In [13]:
text_columns = omi.select_dtypes(include=['object', 'string']).columns

for column in text_columns:
    omi[column] = omi[column].str.strip()

omi[text_columns] = omi[text_columns].replace({'': pd.NA})

print(f'Text columns trimmed: {len(text_columns)}')


Text columns trimmed: 17


In [14]:
omi['reference_date'] = pd.to_datetime({
    'year': omi['reference_year'],
    'month': np.where(omi['reference_semester'].eq('S1'), 6, 12),
    'day': np.where(omi['reference_semester'].eq('S1'), 30, 31),
})

omi = (
    omi
    .sort_values(['reference_date', 'Regione', 'Prov', 'Comune_descrizione'], kind='stable')
    .reset_index(drop=True)
)

print('Cleaned shape:', omi.shape)


Cleaned shape: (7516495, 30)


## 7. Feature Engineering

The raw OMI data provide minimum and maximum quotations. We derive midpoint and spread
measures for descriptive analysis. The midpoint is **not** an official OMI average
price.

**Consistency note.** The midpoint is only computed when **both** bounds are available.
`DataFrame.mean(axis=1)` ignores missing values by default, so a naive
`omi[['Compr_min', 'Compr_max']].mean(axis=1)` would silently fall back to the single
available bound whenever only one of the two is present — treating a one-sided
observation as if it were a genuine midpoint of a range. That is inconsistent with the
spread (`max - min`), which is already strict: it is undefined (`NaN`) whenever either
bound is missing. Requiring both bounds for the midpoint keeps the two measures on the
same footing, and the number of affected rows is reported explicitly.


In [15]:
both_compr_present = omi['Compr_min'].notna() & omi['Compr_max'].notna()
both_loc_present = omi['Loc_min'].notna() & omi['Loc_max'].notna()

print(f"Rows with only one purchase bound available (midpoint set to NaN): {(~both_compr_present & (omi['Compr_min'].notna() | omi['Compr_max'].notna())).sum():,}")
print(f"Rows with only one rental bound available (midpoint set to NaN):   {(~both_loc_present & (omi['Loc_min'].notna() | omi['Loc_max'].notna())).sum():,}")

omi['Compr_mid'] = np.where(both_compr_present, omi[['Compr_min', 'Compr_max']].mean(axis=1), np.nan)
omi['Loc_mid'] = np.where(both_loc_present, omi[['Loc_min', 'Loc_max']].mean(axis=1), np.nan)

omi['Compr_spread'] = omi['Compr_max'] - omi['Compr_min']
omi['Loc_spread'] = omi['Loc_max'] - omi['Loc_min']

omi['Compr_spread_pct'] = omi['Compr_spread'].div(omi['Compr_mid'].replace(0, np.nan)).mul(100)
omi['Loc_spread_pct'] = omi['Loc_spread'].div(omi['Loc_mid'].replace(0, np.nan)).mul(100)

display(
    omi[[
        'Compr_min', 'Compr_max', 'Compr_mid', 'Compr_spread', 'Compr_spread_pct',
        'Loc_min', 'Loc_max', 'Loc_mid', 'Loc_spread', 'Loc_spread_pct'
    ]].describe().T
)


Rows with only one purchase bound available (midpoint set to NaN): 0
Rows with only one rental bound available (midpoint set to NaN):   1,465,886


,count,mean,std,min,25%,50%,75%,max
Compr_min,"7,490,118.00",795.62,617.75,20.00,400.00,640.00,"1,000.00","20,000.00"
Compr_max,"7,490,118.00","1,074.50",848.01,30.00,550.00,850.00,"1,340.00","33,360.00"
Compr_mid,"7,490,118.00",935.06,729.22,25.00,475.00,750.00,"1,160.00","25,020.00"
Compr_spread,"7,490,118.00",278.88,272.83,0.00,130.00,200.00,350.00,"18,000.00"
Compr_spread_pct,"7,490,118.00",30.10,11.05,0.00,22.22,30.77,37.76,176.47
Loc_min,"1,062,446.00",3.66,3.00,1.00,2.00,3.00,5.00,105.00
Loc_max,"1,052,748.00",5.21,4.46,1.00,3.00,4.00,6.00,509.00
Loc_mid,"324,654.00",5.33,4.52,1.00,2.50,3.50,6.50,256.50
Loc_spread,"324,654.00",1.87,2.24,0.00,1.00,1.00,2.00,505.00
Loc_spread_pct,"324,654.00",36.02,12.21,0.00,28.57,40.00,40.00,196.88


## 8. Temporal Coverage

A continuous sequence of semesters is checked explicitly against the full expected range
implied by the earliest and latest year observed. This helps detect incomplete
downloads — a gap here would silently distort semester-over-semester and year-over-year
growth calculations in the exploratory notebook (e.g. `pct_change(2)` for YoY growth
implicitly assumes no gaps).


In [16]:
periods = (
    omi[['reference_year', 'reference_semester', 'reference_period']]
    .drop_duplicates()
    .sort_values(['reference_year', 'reference_semester'])
)

observed_periods = set(periods['reference_period'])
years = range(int(periods['reference_year'].min()), int(periods['reference_year'].max()) + 1)
expected_periods = {f'{year}-S{semester}' for year in years for semester in (1, 2)}
missing_periods = sorted(expected_periods - observed_periods)

print(f'Observed periods:            {len(observed_periods):,}')
print(f'Expected periods in range:   {len(expected_periods):,}')
print('Missing periods:', missing_periods if missing_periods else 'None')


Observed periods:            44
Expected periods in range:   44
Missing periods: None


## 9. Geographic Coverage

The OMI hierarchy allows analysis at several levels: **Area territoriale → Regione →
Provincia → Comune → Zona OMI**.


In [17]:
geographic_summary = pd.DataFrame({
    'regions': [omi['Regione'].nunique()],
    'provinces': [omi['Prov'].nunique()],
    'municipalities': [omi['Comune_ISTAT'].nunique()],
    'omi_zones': [omi['Zona'].nunique()],
})

display(geographic_summary)

regional_coverage = (
    omi
    .groupby('Regione', dropna=False)
    .agg(
        municipalities=('Comune_ISTAT', 'nunique'),
        zones=('Zona', 'nunique'),
        observations=('reference_period', 'size')
    )
    .sort_values('municipalities', ascending=False)
)

display(regional_coverage.head(20))


,regions,provinces,municipalities,omi_zones
0,20,102,8228,393


,municipalities,zones,observations
Regione,,,
LOMBARDIA,1573,122,1199087
PIEMONTE,1223,93,725278
VENETO,595,81,520534
CAMPANIA,552,148,686636
CALABRIA,411,77,381705
SICILIA,392,77,431450
LAZIO,378,339,315825
SARDEGNA,377,41,214284
TRENTINO-ALTO ADIGE,366,53,282401


## 10. Property Categories and Conditions

OMI quotations are segmented by property type (`Descr_Tipologia`) and condition
(`Stato`). Observation counts help distinguish genuine market differences from
differences in survey coverage — a category with very few observations should be
interpreted with more caution than one with thousands.


In [18]:
for column in ['Descr_Tipologia', 'Stato']:
    summary = (
        omi[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name='observations')
    )
    display(summary.head(20))


,Descr_Tipologia,observations
0,Abitazioni civili,1183089
1,Ville e Villini,853424
2,Negozi,788893
3,Box,782999
4,Abitazioni di tipo economico,765963
5,Magazzini,662623
6,Uffici,592632
7,Laboratori,561008
8,Capannoni tipici,428538
9,Capannoni industriali,258369


,Stato,observations
0,NORMALE,6442670
1,OTTIMO,928104
2,NaN,96357
3,SCADENTE,49364


## 11. Final Data-Quality Summary

The final checks provide a compact, auditable hand-off point for the next notebook:
row/column counts, remaining structural-validation flags, and a before/after view of
missingness on the core quotation fields.


In [19]:
final_quality = pd.DataFrame({
    'metric': [
        'rows', 'columns', 'periods', 'duplicate rows',
        'purchase min > max (flagged)', 'rental min > max (flagged)',
        'negative purchase quotations (flagged)', 'negative rental quotations (flagged)'
    ],
    'value': [
        len(omi),
        omi.shape[1],
        omi['reference_period'].nunique(),
        int(omi.duplicated().sum()),
        int(omi['flag_compr_min_gt_max'].sum()),
        int(omi['flag_loc_min_gt_max'].sum()),
        int(omi['flag_negative_compr'].sum()),
        int(omi['flag_negative_loc'].sum())
    ],
})

display(final_quality)


,metric,value
0,rows,7516495
1,columns,36
2,periods,44
3,duplicate rows,0
4,purchase min > max (flagged),0
5,rental min > max (flagged),0
6,negative purchase quotations (flagged),0
7,negative rental quotations (flagged),0


In [20]:
core_fields = ['Compr_min', 'Compr_max', 'Compr_mid', 'Loc_min', 'Loc_max', 'Loc_mid']

missing_after = omi[core_fields].isna().mean().mul(100).rename('missing_pct_after_cleaning')

display(missing_after.to_frame())


,missing_pct_after_cleaning
Compr_min,0.35
Compr_max,0.35
Compr_mid,0.35
Loc_min,85.87
Loc_max,85.99
Loc_mid,95.68


> **Interpretation.** Comparing `missing_pct_after_cleaning` above with the raw
> missingness reported in Section 4.1 shows how much of the core quotation fields'
> missingness is inherent to the source data versus introduced by this pipeline's own
> cleaning decisions (zero-value handling, coercion, strict midpoint calculation).


## 12. Export to Parquet

A final pre-export sanity check, followed by the write itself. The processed directory
is created if it does not already exist, so the export does not fail on a fresh clone of
the repository.


In [21]:
print('Final shape:', omi.shape)
display(omi.head())


Final shape: (7516495, 36)


,Area_territoriale,Regione,Prov,Comune_ISTAT,Comune_cat,Sez,Comune_amm,Comune_descrizione,Fascia,Zona,LinkZona,Cod_Tip,Descr_Tipologia,Stato,Stato_prev,Compr_min,Compr_max,Sup_NL_compr,Loc_min,Loc_max,Sup_NL_loc,reference_year,reference_semester,reference_period,flag_compr_min_gt_max,flag_loc_min_gt_max,flag_negative_compr,flag_negative_loc,flag_invalid_semester,reference_date,Compr_mid,Loc_mid,Compr_spread,Loc_spread,Compr_spread_pct,Loc_spread_pct
0,SUD,ABRUZZO,AQ,"13,066,001.00",N1AB,NaN,A018,ACCIANO,B,B1,AQ00000064,20,Abitazioni civili,NORMALE,NaN,320.00,470.00,L,NaN,NaN,L,2004,S1,2004-S1,False,False,False,False,False,2004-06-30,395.00,NaN,150.00,NaN,37.97,NaN
1,SUD,ABRUZZO,AQ,"13,066,001.00",N1AB,NaN,A018,ACCIANO,B,B1,AQ00000064,21,Abitazioni di tipo economico,NORMALE,NaN,260.00,390.00,L,NaN,1.00,L,2004,S1,2004-S1,False,False,False,False,False,2004-06-30,325.00,NaN,130.00,NaN,40.00,NaN
2,SUD,ABRUZZO,AQ,"13,066,001.00",N1AB,NaN,A018,ACCIANO,B,B1,AQ00000064,13,Box,NaN,NaN,260.00,390.00,L,1.00,NaN,L,2004,S1,2004-S1,False,False,False,False,False,2004-06-30,325.00,NaN,130.00,NaN,40.00,NaN
3,SUD,ABRUZZO,AQ,"13,066,001.00",N1AB,NaN,A018,ACCIANO,B,B1,AQ00000064,14,Posti auto coperti,NaN,NaN,170.00,250.00,L,NaN,1.00,L,2004,S1,2004-S1,False,False,False,False,False,2004-06-30,210.00,NaN,80.00,NaN,38.10,NaN
4,SUD,ABRUZZO,AQ,"13,066,001.00",N1AB,NaN,A018,ACCIANO,B,B1,AQ00000064,15,Posti auto scoperti,NaN,NaN,70.00,100.00,L,NaN,NaN,L,2004,S1,2004-S1,False,False,False,False,False,2004-06-30,85.00,NaN,30.00,NaN,35.29,NaN


In [22]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / 'omi_quotations.parquet'
omi.to_parquet(output_path, index=False)

print(f'Dataset exported to: {output_path}')


Dataset exported to: c:\Users\cc1323\italian-real-estate-analysis\data\processed\omi_quotations.parquet
